<a href="https://colab.research.google.com/github/hamza-24-ai/ca_research/blob/main/CA_Task_03_Transformer_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer Testing DistilBert

## Testing by CPU

In [ ]:
!pip -q install transformers datasets

import copy, time, statistics, csv, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

torch.manual_seed(0)
dev = torch.device("cpu")
MODEL = "distilbert-base-uncased-finetuned-sst-2-english"
SEQ = 128
print("PyTorch", torch.__version__, "| device:", dev, "| model:", MODEL)

# ---------- load (startup) ----------
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(MODEL)
base = AutoModelForSequenceClassification.from_pretrained(MODEL).eval()
t_load = time.perf_counter() - t0
params = sum(p.numel() for p in base.parameters())
print("load_s:", round(t_load, 3), "| params:", params)

# ---------- fixed input for speed tests (synthetic ids, fixed shape) ----------
def make_inputs(bs):
    return {"input_ids": torch.randint(1000, 20000, (bs, SEQ)),
            "attention_mask": torch.ones(bs, SEQ, dtype=torch.long)}

# ---------- FLOPs estimate (per sequence, 128 tokens) ----------
from torch.utils.flop_counter import FlopCounterMode
with FlopCounterMode(display=False) as fc:
    with torch.inference_mode():
        base(**make_inputs(1))
flops_per_seq = fc.get_total_flops()
print("FLOPs/seq:", flops_per_seq)

# ---------- first forward (startup) ----------
t1 = time.perf_counter()
with torch.inference_mode():
    base(**make_inputs(1))
t_first = time.perf_counter() - t1
print("first_forward_s:", round(t_first, 4))

# ---------- speed ----------
def bench(m, dtype_name, dtype, bs, reps):
    x = {k: v for k, v in make_inputs(bs).items()}
    with torch.inference_mode():
        for _ in range(3):
            m(**x)
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter(); m(**x); ts.append(time.perf_counter() - t0)
    s = sorted(ts); med = statistics.median(ts)
    return {"device": "cpu", "precision": dtype_name, "batch": bs, "reps": reps,
            "median_s": med, "std_s": statistics.stdev(ts),
            "p95_s": s[int(0.95*(len(s)-1))], "p99_s": s[int(0.99*(len(s)-1))],
            "throughput_seq_s": bs / med,
            "gflops_per_s": flops_per_seq * bs / med / 1e9, "status": "ok"}

rows = []
for name, dt in [("fp32", torch.float32), ("bf16", torch.bfloat16)]:
    m = copy.deepcopy(base).to(dt)
    for bs in [1, 2, 4, 8, 16, 32]:
        try:
            rows.append(bench(m, name, dt, bs, 100 if bs == 1 else 10))
        except Exception as e:
            rows.append({"device": "cpu", "precision": name, "batch": bs,
                         "status": f"failed: {type(e).__name__}: {str(e)[:100]}"})
        print(rows[-1])

# ---------- accuracy on SST-2 validation (fixed 872 examples, or 1000 max) ----------
val = load_dataset("glue", "sst2", split="validation")
val = val.shuffle(seed=0).select(range(min(1000, len(val))))
print("accuracy set size:", len(val))

def accuracy(m, dtype):
    correct = 0
    with torch.inference_mode():
        for i in range(0, len(val), 32):
            batch = val[i:i+32]
            enc = tok(batch["sentence"], padding="max_length", truncation=True,
                      max_length=SEQ, return_tensors="pt")
            logits = m(**enc).logits.float()
            correct += (logits.argmax(1) == torch.tensor(batch["label"])).sum().item()
    return correct / len(val)

for name, dt in [("fp32", torch.float32), ("bf16", torch.bfloat16)]:
    m = copy.deepcopy(base).to(dt)
    acc = accuracy(m, dt)
    rows.append({"device": "cpu", "precision": name, "batch": "accuracy",
                 "reps": len(val), "accuracy": acc, "status": "ok"})
    print(name, "accuracy:", acc)

with open("distilbert_cpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r}))
    w.writeheader(); w.writerows(rows)
print("Saved distilbert_cpu.csv")

PyTorch 2.14.1+cpu | device: cpu | model: distilbert-base-uncased-finetuned-sst-2-english


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

load_s: 25.894 | params: 66955010
FLOPs/seq: 10872818688
first_forward_s: 0.7987
{'device': 'cpu', 'precision': 'fp32', 'batch': 1, 'reps': 100, 'median_s': 0.23804193399999463, 'std_s': 0.05872600205438278, 'p95_s': 0.37546035399998345, 'p99_s': 0.38843636699999706, 'throughput_seq_s': 4.200940494795436, 'gflops_per_s': 45.676064318987784, 'status': 'ok'}
{'device': 'cpu', 'precision': 'fp32', 'batch': 2, 'reps': 10, 'median_s': 0.4440723440000056, 'std_s': 0.11191221750396178, 'p95_s': 0.6951416039999572, 'p99_s': 0.6951416039999572, 'throughput_seq_s': 4.503770673906175, 'gflops_per_s': 48.96868194971341, 'status': 'ok'}
{'device': 'cpu', 'precision': 'fp32', 'batch': 4, 'reps': 10, 'median_s': 0.8814362614999709, 'std_s': 0.23603984392341895, 'p95_s': 1.2841750380000008, 'p99_s': 1.2841750380000008, 'throughput_seq_s': 4.538047927813929, 'gflops_per_s': 49.34137231657497, 'status': 'ok'}
{'device': 'cpu', 'precision': 'fp32', 'batch': 8, 'reps': 10, 'median_s': 1.5848795899999857, 

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

HfUriError: Invalid HF URI 'hf://datasets/glue@bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/.huggingface.yaml'. Repository id must be 'namespace/name', got 'glue'.

## Accuracy ON CPU

In [ ]:
!pip -q install datasets

import copy, csv, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

torch.manual_seed(0)
MODEL = "distilbert-base-uncased-finetuned-sst-2-english"
SEQ = 128
tok = AutoTokenizer.from_pretrained(MODEL)
base = AutoModelForSequenceClassification.from_pretrained(MODEL).eval()

val = load_dataset("nyu-mll/glue", "sst2", split="validation")
val = val.shuffle(seed=0).select(range(min(1000, len(val))))
print("accuracy set size:", len(val))

def accuracy(m, dtype):
    correct = 0
    with torch.inference_mode():
        for i in range(0, len(val), 32):
            batch = val[i:i+32]
            enc = tok(batch["sentence"], padding="max_length", truncation=True,
                      max_length=SEQ, return_tensors="pt")
            logits = m(**enc).logits.float()
            correct += (logits.argmax(1) == torch.tensor(batch["label"])).sum().item()
    return correct / len(val)

rows = []
for name, dt in [("fp32", torch.float32), ("bf16", torch.bfloat16)]:
    m = copy.deepcopy(base).to(dt)
    acc = accuracy(m, dt)
    rows.append({"device": "cpu", "precision": name, "batch": "accuracy",
                 "reps": len(val), "accuracy": acc})
    print(name, "accuracy:", acc)

with open("distilbert_cpu_accuracy.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print("Saved distilbert_cpu_accuracy.csv")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (2 outputs left out)

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

accuracy set size: 872
fp32 accuracy: 0.9105504587155964
bf16 accuracy: 0.9105504587155964
Saved distilbert_cpu_accuracy.csv


## GPU Model Testing

In [ ]:
!pip -q install transformers datasets

import copy, time, statistics, csv, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

torch.manual_seed(0)
dev = torch.device("cuda")
MODEL = "distilbert-base-uncased-finetuned-sst-2-english"
SEQ = 128
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0), "| model:", MODEL)

def sync():
    torch.cuda.synchronize()

# ---------- load (startup) ----------
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(MODEL)
base = AutoModelForSequenceClassification.from_pretrained(MODEL).eval()
t_load = time.perf_counter() - t0
params = sum(p.numel() for p in base.parameters())
print("load_s:", round(t_load, 3), "| params:", params)

# ---------- first forward (startup, CUDA already initialized in this session) ----------
def make_inputs(bs):
    return {"input_ids": torch.randint(1000, 20000, (bs, SEQ), device=dev),
            "attention_mask": torch.ones(bs, SEQ, dtype=torch.long, device=dev)}

base = base.to(dev)
t1 = time.perf_counter()
with torch.inference_mode():
    base(**make_inputs(1))
sync()
t_first = time.perf_counter() - t1
print("first_forward_s:", round(t_first, 4))

# ---------- FLOPs estimate (per sequence, 128 tokens) ----------
from torch.utils.flop_counter import FlopCounterMode
with FlopCounterMode(display=False) as fc:
    with torch.inference_mode():
        base(**make_inputs(1))
flops_per_seq = fc.get_total_flops()
print("FLOPs/seq:", flops_per_seq)

# ---------- speed ----------
def bench(m, dtype_name, dtype, bs, reps):
    x = make_inputs(bs)
    with torch.inference_mode():
        for _ in range(3):
            m(**x)
        sync()
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter(); m(**x); sync(); ts.append(time.perf_counter() - t0)
    s = sorted(ts); med = statistics.median(ts)
    return {"device": "cuda", "precision": dtype_name, "batch": bs, "reps": reps,
            "median_s": med, "std_s": statistics.stdev(ts),
            "p95_s": s[int(0.95*(len(s)-1))], "p99_s": s[int(0.99*(len(s)-1))],
            "throughput_seq_s": bs / med,
            "gflops_per_s": flops_per_seq * bs / med / 1e9, "status": "ok"}

rows = []
for name, dt in [("fp32", torch.float32), ("fp16", torch.float16), ("bf16", torch.bfloat16)]:
    m = copy.deepcopy(base).to(dt)
    for bs in [1, 2, 4, 8, 16, 32]:
        try:
            rows.append(bench(m, name, dt, bs, 100 if bs == 1 else 10))
        except Exception as e:
            rows.append({"device": "cuda", "precision": name, "batch": bs,
                         "status": f"failed: {type(e).__name__}: {str(e)[:100]}"})
        print(rows[-1])

# ---------- accuracy on SST-2 validation (all 872 examples, same as CPU) ----------
val = load_dataset("nyu-mll/glue", "sst2", split="validation")
val = val.shuffle(seed=0).select(range(min(1000, len(val))))
print("accuracy set size:", len(val))

def accuracy(m):
    correct = 0
    with torch.inference_mode():
        for i in range(0, len(val), 32):
            batch = val[i:i+32]
            enc = tok(batch["sentence"], padding="max_length", truncation=True,
                      max_length=SEQ, return_tensors="pt").to(dev)
            logits = m(**enc).logits.float().cpu()
            correct += (logits.argmax(1) == torch.tensor(batch["label"])).sum().item()
    return correct / len(val)

for name, dt in [("fp32", torch.float32), ("fp16", torch.float16), ("bf16", torch.bfloat16)]:
    m = copy.deepcopy(base).to(dt)
    acc = accuracy(m)
    rows.append({"device": "cuda", "precision": name, "batch": "accuracy",
                 "reps": len(val), "accuracy": acc, "status": "ok"})
    print(name, "accuracy:", acc)

with open("distilbert_gpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r}))
    w.writeheader(); w.writerows(rows)
print("Saved distilbert_gpu.csv")

PyTorch 2.14.1+cu130 | GPU: Tesla T4 | model: distilbert-base-uncased-finetuned-sst-2-english


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

... (9 outputs left out)

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

accuracy set size: 872
fp32 accuracy: 0.9105504587155964
fp16 accuracy: 0.9105504587155964
bf16 accuracy: 0.9094036697247706
Saved distilbert_gpu.csv


## TPU Testing

In [1]:
!pip -q install transformers datasets

import copy, time, statistics, csv, torch, torch_tpu
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

torch.manual_seed(0)
dev = torch.device("tpu")
MODEL = "distilbert-base-uncased-finetuned-sst-2-english"
SEQ = 128
print("PyTorch", torch.__version__, "| device:", dev, "| model:", MODEL)

def sync(out):
    out.sum().item()      # forces the TPU to finish, brings a value back to host

# ---------- load (startup) ----------
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(MODEL)
base = AutoModelForSequenceClassification.from_pretrained(MODEL).eval()
t_load = time.perf_counter() - t0
params = sum(p.numel() for p in base.parameters())
print("load_s:", round(t_load, 3), "| params:", params)

def make_inputs(bs):
    return {"input_ids": torch.randint(1000, 20000, (bs, SEQ), device=dev),
            "attention_mask": torch.ones(bs, SEQ, dtype=torch.long, device=dev)}

# ---------- FLOPs estimate on CPU copy (same model, same shape) ----------
from torch.utils.flop_counter import FlopCounterMode
with FlopCounterMode(display=False) as fc:
    with torch.inference_mode():
        base(input_ids=torch.randint(1000, 20000, (1, SEQ)),
             attention_mask=torch.ones(1, SEQ, dtype=torch.long))
flops_per_seq = fc.get_total_flops()
print("FLOPs/seq:", flops_per_seq)

# ---------- move model, first forward (includes XLA compile) ----------
base = base.to(dev)
t1 = time.perf_counter()
with torch.inference_mode():
    sync(base(**make_inputs(1)).logits)
t_first = time.perf_counter() - t1
print("first_forward_s (incl. compile):", round(t_first, 4))

# ---------- speed ----------
compile_times = {}
def bench(m, dtype_name, dtype, bs, reps):
    x = make_inputs(bs)
    with torch.inference_mode():
        t0 = time.perf_counter(); sync(m(**x).logits)      # first call at this shape = compile
        compile_times[(dtype_name, bs)] = time.perf_counter() - t0
        for _ in range(2):
            sync(m(**x).logits)
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter(); out = m(**x).logits; sync(out); ts.append(time.perf_counter() - t0)
    s = sorted(ts); med = statistics.median(ts)
    return {"device": "tpu", "precision": dtype_name, "batch": bs, "reps": reps,
            "median_s": med, "std_s": statistics.stdev(ts),
            "p95_s": s[int(0.95*(len(s)-1))], "p99_s": s[int(0.99*(len(s)-1))],
            "throughput_seq_s": bs / med,
            "gflops_per_s": flops_per_seq * bs / med / 1e9,
            "compile_s": compile_times[(dtype_name, bs)], "status": "ok"}

rows = []
for name, dt in [("fp32", torch.float32), ("fp16", torch.float16), ("bf16", torch.bfloat16)]:
    m = copy.deepcopy(base).to(dt).to(dev)
    for bs in [1, 2, 4, 8, 16, 32]:
        try:
            rows.append(bench(m, name, dt, bs, 100 if bs == 1 else 10))
        except Exception as e:
            rows.append({"device": "tpu", "precision": name, "batch": bs,
                         "status": f"failed: {type(e).__name__}: {str(e)[:100]}"})
        print(rows[-1])

# ---------- accuracy on SST-2 validation (all 872 examples, same as CPU and GPU) ----------
val = load_dataset("nyu-mll/glue", "sst2", split="validation")
val = val.shuffle(seed=0).select(range(min(1000, len(val))))
print("accuracy set size:", len(val))

def accuracy(m):
    correct = 0
    with torch.inference_mode():
        for i in range(0, len(val), 32):
            batch = val[i:i+32]
            enc = tok(batch["sentence"], padding="max_length", truncation=True,
                      max_length=SEQ, return_tensors="pt")
            enc = {k: v.to(dev) for k, v in enc.items()}
            logits = m(**enc).logits.float().cpu()
            correct += (logits.argmax(1) == torch.tensor(batch["label"])).sum().item()
    return correct / len(val)

for name, dt in [("fp32", torch.float32), ("fp16", torch.float16), ("bf16", torch.bfloat16)]:
    try:
        m = copy.deepcopy(base).to(dt).to(dev)
        acc = accuracy(m)
        rows.append({"device": "tpu", "precision": name, "batch": "accuracy",
                     "reps": len(val), "accuracy": acc, "status": "ok"})
        print(name, "accuracy:", acc)
    except Exception as e:
        rows.append({"device": "tpu", "precision": name, "batch": "accuracy",
                     "status": f"failed: {type(e).__name__}: {str(e)[:100]}"})

with open("distilbert_tpu.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r}))
    w.writeheader(); w.writerows(rows)
print("Saved distilbert_tpu.csv")

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


PyTorch 2.14.1+cpu | device: tpu | model: distilbert-base-uncased-finetuned-sst-2-english


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

... (12 outputs left out)

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

accuracy set size: 872
fp32 accuracy: 0.9105504587155964
fp16 accuracy: 0.9105504587155964
bf16 accuracy: 0.9094036697247706
Saved distilbert_tpu.csv
